## 2つのモチーフ、1つの遺伝子

「共有されたスプライスモチーフを見つける」では、2つの遺伝子文字列に共通して含まれていた可能性のある最長のモチーフを探しました。このとき、モチーフは複数のエクソンに分かれて存在していてもよいものとしました。そのため、2つの文字列の最長共通部分列を見つける必要がありました。これは、「共有モチーフを見つける」で扱った最長共通部分文字列の問題を拡張したものです。

この問題では、ある意味でその逆問題を考えます。つまり、2つの異なるモチーフが与えられたとき、それらをうまく組み合わせて、両方のモチーフが部分列として現れるような、できるだけ短い遺伝子文字列を作ることを目指します。

## 問題

文字列 (s) が別の文字列 (t) の**スーパーシーケンス**であるとは、(s) が (t) を部分列として含むことをいいます。

文字列 (s) と (t) の**共通スーパーシーケンス**とは、(s) と (t) の両方に対するスーパーシーケンスとなる文字列のことです。

例えば、`GACCTAGGAACTC` は、`ACGTC` と `ATAT` の共通スーパーシーケンスです。

(s) と (t) の**最短共通スーパーシーケンス**とは、それより短い共通スーパーシーケンスが存在しないような共通スーパーシーケンスのことです。先ほどの例では、`ACGTACT` が `ACGTC` と `ATAT` の最短共通スーパーシーケンスです。

## 与えられるもの

2つのDNA文字列 (s) と (t)。

## 返すもの

(s) と (t) の最短共通スーパーシーケンスを1つ返してください。複数の解が存在する場合は、そのうち任意の1つを出力してかまいません。


### 考え方
2つの配列の先頭を比較し  
同じ文字列であれば両方の文字列から文字を進めて、共通配列に追加  
違う場合には、どちらかの文字を共通配列に追加して、もう一方の文字列はそのままにして比較を続ける。

↑これだとうまくいかない。

In [1]:
sample_imput = """ATCTGAT
TGCATA"""

In [2]:
def parse_input(input_string):
    lines = input_string.strip().splitlines()
    return lines[0], lines[1]

In [3]:
def main(s1, s2):
    i, j = 0, 0
    result = []
    while i < len(s1) and j < len(s2):
        if s1[i] == s2[j]:
            result.append(s1[i])
            i += 1
            j += 1
        else:
            result.append(s1[i])
            i += 1
    # Append remaining characters from s1 or s2)
    result.extend(s2[j:])
    return ''.join(result)

In [4]:
parse_input(sample_imput)

('ATCTGAT', 'TGCATA')

In [5]:
# できれば同時に歩みを進めたいが、これだと3歩めのCが無駄

main(*parse_input(sample_imput))

'ATCTGATCATA'

### 新しい考え方
最長部分共通配列を求める。
↑の配列の前後にs1 or s2の文字列を追加することで、最短共通スーパーシーケンスを作ることができる。

from 37-LCSQ

| `dp` | `""` | `T` | `G` | `C` | `A` | `T` | `A` |
| ---- | ---: | --: | --: | --: | --: | --: | --: |
| `""` |    0 |   0 |   0 |   0 |   0 |   0 |   0 |
| `A`  |    0 |   0 |   0 |   0 |   1 |   1 |   1 |
| `T`  |    0 |   1 |   1 |   1 |   1 |   2 |   2 |
| `C`  |    0 |   1 |   1 |   2 |   2 |   2 |   2 |
| `T`  |    0 |   1 |   1 |   2 |   2 |   3 |   3 |
| `G`  |    0 |   1 |   2 |   2 |   2 |   3 |   3 |
| `A`  |    0 |   1 |   2 |   2 |   3 |   3 |   4 |
| `T`  |    0 |   1 |   2 |   2 |   3 |   4 |   4 |

**数字の埋め方**
文字が一致する場合は、その文字を使えるので、左上の値に1を足す。
文字が一致しない場合は、左か上のどちらかの値を使う。大きい方が長い共通部分列になる。

**戻し方**
右下より、
1. 文字が同じなら左上、
2. 文字が違うなら、左に進む
を繰り返すことで、最長共通部分列を求めることができる。


In [6]:
def lcs(seq1:str, seq2: str) -> str:
    m, n = len(seq1), len(seq2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[i - 1] == seq2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    
    lcs_str = []
    i, j = m, n
    while i > 0 and j > 0:
        if seq1[i - 1] == seq2[j - 1]:
            lcs_str.append(seq1[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] > dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    
    return ''.join(reversed(lcs_str))

In [7]:
parse_input(sample_imput)

('ATCTGAT', 'TGCATA')

In [8]:
lcs(*parse_input(sample_imput))

'TGAT'

In [9]:
def answer(s1, s2):
    lcs_str = lcs(s1, s2)
    i, j = 0, 0
    result = []
    
    for c in lcs_str:
        while i < len(s1) and s1[i] != c:
            result.append(s1[i])
            i += 1
        while j < len(s2) and s2[j] != c:
            result.append(s2[j])
            j += 1
        result.append(c)
        i += 1
        j += 1
    
    result.extend(s1[i:])
    result.extend(s2[j:])
    
    return ''.join(result)

In [10]:
answer(*parse_input(sample_imput))

'ATCTGCATA'

### 具体例
`ATCTGAT` と `TGCATA` の最長共通部分列は、例えば `TGAT` です。

```text
s1: A T C T G A T
      T     G A T

s2: T G C A T A
    T G   A T
```

`TGAT` は両方の配列に同じ順番で含まれているため、最短共通スーパーシーケンスではこの部分を1回だけ使えます。

あとは、それぞれの配列で `TGAT` の間にある文字を順番に足します。

```text
s1由来: A CT
共有:    T  G AT
s2由来:      C  A
```

したがって、最短共通スーパーシーケンスの一例は次のようになります。

```text
ATCTGCATA
```